In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

soil_path = "../data/raw/state_soil_data.csv"
weather_path = "../data/raw/state_weather_data_1997_2020.csv"

soil = pd.read_csv(soil_path)
weather = pd.read_csv(weather_path)

print("Soil shape:", soil.shape)
print("Weather shape:", weather.shape)
soil.head(), weather.head()

In [ ]:
# Basic info and missing values
display(soil.info())
display(weather.info())
soil.isna().sum(), weather.isna().sum()

In [ ]:
# Clean whitespace and standardize state names
for df in (soil, weather):
    df["state"] = df["state"].astype(str).str.strip()

# Ensure numeric types for soil
for col in ["N", "P", "K", "pH"]:
    soil[col] = pd.to_numeric(soil[col], errors="coerce")

# Ensure numeric types for weather
for col in ["year", "avg_temp_c", "total_rainfall_mm", "avg_humidity_percent"]:
    weather[col] = pd.to_numeric(weather[col], errors="coerce")

soil.describe(include="all").T, weather.describe(include="all").T

In [ ]:
# Soil feature distributions
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flatten(), ["N", "P", "K", "pH"]):
    sns.histplot(soil[col].dropna(), bins=20, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

print("States with highest soil N:")
display(soil.sort_values("N", ascending=False).head(10)[["state", "N"]])

In [ ]:
# Weather trends: rainfall and temperature over time (state-level averages)
state_year_weather = weather.groupby(["state", "year"], as_index=False).mean()

example_states = ["Punjab", "Maharashtra", "Tamil Nadu", "Assam"]
subset = state_year_weather[state_year_weather["state"].isin(example_states)]

plt.figure(figsize=(10, 5))
sns.lineplot(data=subset, x="year", y="total_rainfall_mm", hue="state")
plt.title("Total rainfall over time for selected states")
plt.show()

plt.figure(figsize=(10, 5))
sns.lineplot(data=subset, x="year", y="avg_temp_c", hue="state")
plt.title("Average temperature over time for selected states")
plt.show()

state_year_weather.head()

In [ ]:
# Save cleaned/standardized versions
import os
os.makedirs("../data/processed", exist_ok=True)

soil_clean_path = "../data/processed/state_soil_data_clean.csv"
weather_clean_path = "../data/processed/state_weather_data_clean.csv"

soil.to_csv(soil_clean_path, index=False)
weather.to_csv(weather_clean_path, index=False)

print("Saved cleaned soil data to", soil_clean_path)
print("Saved cleaned weather data to", weather_clean_path)